# Capítulo 4 — Flujo de gas por tuberías (secciones 4.1 a 4.4)

> **Fuente.** Métodos y datos de *Transporte de Hidrocarburos por Ductos*, F. Garaicochea P.,
> C. Bernal H. y O. López O. (CIPM, 1991; reed. 2022), verificados contra la tesis UNAM de 1990 que
> le dio origen (Bernal y López, asesor Garaicochea). Los enunciados están **parafraseados**; consulte la
> obra original para el texto completo.

Las secciones 4.5 a 4.10 (condensados, costos y diseño económico, pozo productor, estranguladores y raspatubos)
están en `04b_gas_economia_pozos_estranguladores`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ductos import friccion as fr
from ductos.constantes import c_general_gas
from ductos.pvt_gas import Gas
from ductos.gas import (Gasoducto, ECUACIONES, p2_general, presion_media, presion_fondo_estatica,
                        q_panhandle_b, factor_s, longitud_efectiva, longitud_equivalente_serie,
                        diametro_equivalente_paralelo, fraccion_loop, q_general)

## 4.1 Ecuación general de energía en unidades prácticas

Para flujo permanente, una sola fase y energía cinética despreciable:

$$p_1^2 - e^s p_2^2 = \frac{f\,q^2\,\gamma_g\,\bar Z\,\bar T\,L_e}{C^2\,d^5}\left(\frac{p_b}{T_b}\right)^2,\qquad s = \frac{0.0375\,\gamma_g\,h}{\bar Z\,\bar T}$$

La constante $C$ sale de Darcy-Weisbach con la ley de los gases reales:

In [ ]:
print(f"C = {c_general_gas():.3f}   (literatura: 77.54)")

### Ejemplo 4.1: presión de fondo de un pozo de gas cerrado

**Planteamiento (parafraseado).** Pozo de 6320 pies, presión en la cabeza 1920 lb/pg² abs, temperatura media 140 °F,
gas de densidad relativa 0.85. Sin flujo solo actúa la columna de gas (Ec. 4.18), que se resuelve por ensaye
y error porque Z depende de la presión media.

In [ ]:
for media in ("aritmetica", "cubica"):
    r = presion_fondo_estatica(1920, 6320, 140, Gas(0.85), media=media)
    print(f"p̄ {media:10s}: pws = {r['pws']:7.1f} lb/pg²  (Z = {r['Z']:.4f}, {r['iteraciones']} iter.)")
print("Libro: 2459.3 lb/pg²")

El texto describe la media aritmética, pero el resultado publicado coincide mejor con la presión media de la Ec. 4.46.
Diferencia entre ambas: 0.25 %.

## 4.2 Número de Reynolds para gas

$$N_{Re} = 0.0201056\,\frac{q\,\gamma_g}{d\,\mu_g}$$

Con gastos altos y viscosidades bajas el flujo es casi siempre francamente turbulento, por eso el libro
usa la Ec. 2.18 para $f$ (sección 4.3.1).

In [ ]:
from ductos.gas import reynolds
g = Gas(0.6)
mu = g.viscosidad(1707.84, 150)
nre = reynolds(1212.56e6, 0.6, 28, mu)
print(f"μg = {mu:.4f} cp,  NRe = {nre:.3e}")
print(f"f Colebrook = {fr.f_colebrook(nre, 0.0006/28):.6f}   f rugoso (Ec. 2.18) = {fr.f_rugoso(0.0006/28):.6f}")

## Ejemplo 4.2: presión de descarga y comparación de ecuaciones

**Planteamiento (parafraseado).** Gasoducto horizontal de 28 pg y 41 millas, ε = 0.0006 pg, gas de γ = 0.6,
150 °F, condiciones base 14.7 lb/pg² y 60 °F. Entra a 1900 lb/pg² abs con 1212.557807 MMpie³/día. Calcular la
presión de llegada y comparar el gasto que predice cada ecuación de la Tabla 4.1.

In [ ]:
gas = Gas(0.6)
ducto = Gasoducto(d=28, L=41, t_f=150)
r = p2_general(1900, 1212.557807e6, gas, ducto)
print(f"p2 = {r['p2']:.1f} lb/pg² abs   p̄ = {r['p_media']:.2f}   Z = {r['Z']:.5f}   f = {r['f']:.7f}")
print("Libro: p2 = 1500, p̄ = 1707.84, Z = 0.87695, f = 0.0091085")

In [ ]:
libro = {"General (4.20)": 1212.558, "Panhandle A": 1472.509, "Panhandle B": 1721.002,
         "Weymouth": 1126.940, "Smith": 1216.592, "Clinedinst": 1213.109, "Cullender-Smith": 1216.335}
q_ref = 1212.557807
filas = []
for nombre, fn in ECUACIONES.items():
    q = fn(1900, 1500, gas, ducto) / 1e6
    filas.append({"ecuación": nombre, "q (MMpcd)": q, "libro": libro[nombre],
                  "desviación vs. general %": 100 * (q - q_ref) / q_ref})
qb = q_panhandle_b(1900, 1500, gas, ducto, variante="libro") / 1e6
filas.append({"ecuación": "Panhandle B (variante libro)", "q (MMpcd)": qb, "libro": 1721.002,
              "desviación vs. general %": 100 * (qb - q_ref) / q_ref})
tabla = pd.DataFrame(filas).set_index("ecuación").round(3)
tabla

Puntos para clase:

* **Panhandle B.** El valor publicado (1721 MMpcd, +42 %) sale de la constante $K_4$ de la Tabla 4.3, que deja
  $\gamma^{0.961}$ fuera del exponente 0.51. Con la forma de la literatura el gasto queda en 1353 MMpcd.
  Se conservan ambas formas (`variante="libro"`); ver `docs/erratas.md`.
* **Clinedinst** integra $p_{pr}/Z$ en lugar de usar un Z medio. El libro toma la integral de tablas y aquí se
  calcula numéricamente con DPR; la diferencia es de 0.08 %.
* **Cullender-Smith** (Apéndice D) evalúa Z y T en cada punto en lugar de usar un Z medio; en una línea
  horizontal integra $p/ZT$ con dos trapecios.
* **Weymouth** usa $f = 0.032/d^{1/3}$, que para 28 pg es más conservador que la Ec. 2.18.
* Los factores de eficiencia que hacen coincidir cada ecuación con el gasto de referencia son simplemente
  $q_{ref}/q_{ec}$.

### Perfil de presión a lo largo del gasoducto

La distribución no es lineal, por eso se usa la presión media de la Ec. 4.46.

In [ ]:
xs = np.linspace(0, 41, 42)
ps = [1900] + [p2_general(1900, 1212.557807e6, gas, Gasoducto(28, x, 150))["p2"] for x in xs[1:]]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, ps, label="perfil calculado")
ax.plot([0, 41], [1900, ps[-1]], "--", c="0.5", label="lineal")
ax.axhline(presion_media(1900, ps[-1]), ls=":", c="C3", label=f"p̄ Ec. 4.46 = {presion_media(1900, ps[-1]):.0f}")
ax.set(xlabel="Distancia (millas)", ylabel="Presión (lb/pg² abs)", title="Ejemplo 4.2 — perfil de presión")
ax.grid(alpha=0.3); ax.legend(); plt.show()

## Ejemplo 4.3: tubería inclinada

Mismos datos que el 4.2, con la descarga 50 pies arriba de la entrada. Se corrige con $e^s$ y la longitud efectiva (Ecs. 4.23-4.24).

In [ ]:
s = factor_s(0.6, 50, 0.87695, 150)
print(f"s = {s:.6f}   e^s = {np.exp(s):.7f}   Le = {longitud_efectiva(41, s):.3f} millas   [libro: 0.002103, 41.043]")
inclinado = Gasoducto(d=28, L=41, t_f=150, h=50)
libro_43 = {"General (4.20)": 1209.189, "Panhandle A": 1468.912, "Panhandle B": 1717.025,
            "Weymouth": 1124.389, "Smith": 1213.879, "Cullender-Smith": 1215.572}
filas = []
for nombre, ref in libro_43.items():
    fn = ECUACIONES[nombre]
    q = (q_panhandle_b(1900, 1500, gas, inclinado, variante="libro") if nombre == "Panhandle B"
         else fn(1900, 1500, gas, inclinado)) / 1e6
    filas.append({"ecuación": nombre, "q (MMpcd)": q, "libro": ref})
pd.DataFrame(filas).set_index("ecuación").round(3)

## 4.4 Sistemas complejos

### 4.4.1 Tuberías en serie — Ejemplo 4.4

Tres tramos: 10 millas de 14 pg, 10 millas de 16 pg y 15 millas de 18 pg, ε = 0.0006 pg, flujo completamente
turbulento. Longitud equivalente referida a 16 pg.

Con todo lo demás constante, $p_1^2-p_2^2 \propto f L/d^5$, de modo que

$$L_E = \frac{d_{ref}^5}{f_{ref}}\sum_i \frac{L_i f_i}{d_i^5}$$

In [ ]:
tramos = [(10, 14, fr.f_rugoso(0.0006 / 14)), (10, 16, fr.f_rugoso(0.0006 / 16)), (15, 18, fr.f_rugoso(0.0006 / 18))]
for L, d, f in tramos:
    print(f"{d} pg: f = {f:.6f}")
f16 = tramos[1][2]
le = longitud_equivalente_serie(tramos, 16, f16)
print(f"\nL_E (d⁵, f) = {le:.2f} millas")
le_libro = 16**2.5 / f16 * sum(L * f / d**2.5 for L, d, f in tramos if d != 16) + 10
print(f"L_E con la forma que aparece en la fuente (d^2.5 y f^1): {le_libro:.2f} millas  [libro: 27.92]")

**Discusión.** La fuente combina $d^{2.5}$ con $f^{1}$ (su factor $16^{2.5}/0.010016 = 102\,236$ lo confirma), lo cual no
conserva $p_1^2-p_2^2$. Además, en el término del tramo de 18 pg anota $35.4\times10^{-6}$ cuando
$15(0.009814)/18^{2.5} = 107.1\times10^{-6}$: un error aritmético que explica la diferencia restante hasta sus 27.92 millas.
La prueba decisiva es física: el gasto que pasa por la serie real debe ser igual al de la línea equivalente.

In [ ]:
p1, p2, T = 1900, 1500, 150
def q_serie(p1, p2):
    # gasto que cumple la suma de Δp² en los tres tramos (Z y T medios constantes)
    z = gas.z(presion_media(p1, p2), T)
    suma = sum(L * f / d**5 for L, d, f in tramos)
    return c_general_gas() * 520 / 14.7 * np.sqrt((p1**2 - p2**2) / (gas.gamma * z * (T + 460) * suma))
print(f"q serie real           = {q_serie(p1, p2)/1e6:8.2f} MMpcd")
for L in (le, le_libro):
    print(f"q línea 16 pg, {L:5.2f} mi = {q_general(p1, p2, gas, Gasoducto(16, L, T), f=f16)/1e6:8.2f} MMpcd")

### 4.4.2 Tuberías en paralelo — Ejemplo 4.5

En paralelo la caída es la misma y los gastos se suman; como $q \propto d^{2.5}/\sqrt{f}$:

$$\frac{d_E^{2.5}}{\sqrt{f_E}} = \sum_i \frac{d_i^{2.5}}{\sqrt{f_i}}$$

**Datos.** Dos líneas paralelas de 14 y 16 pg a lo largo de 30 millas, seguidas de 70 millas de 16 pg.
La configuración se reconstruyó a partir de los números del ejemplo, porque la Fig. 4.2 no es legible en la tesis
escaneada; conviene cotejarla con el libro impreso.

In [ ]:
ramas = [(14, fr.f_rugoso(0.0006 / 14)), (16, fr.f_rugoso(0.0006 / 16))]
d_e, f_e = diametro_equivalente_paralelo(ramas)
print(f"d_E = {d_e:.2f} pg  (f_E = {f_e:.6f})")
le_16 = longitud_equivalente_serie([(30, d_e, f_e), (70, 16, f16)], 16, f16)
le_de = longitud_equivalente_serie([(30, d_e, f_e), (70, 16, f16)], d_e, f_e)
print(f"Sistema equivalente: {le_16:.2f} millas de 16 pg, o {le_de:.2f} millas de {d_e:.2f} pg")

# Lo que hace la fuente: pondera ambas líneas con (f14/f16)^0.5 y usa d^2.5 en la serie
f14 = ramas[0][1]
de_libro = ((f14 / f16)**0.5 * (14**2.5 + 16**2.5))**0.4
le_libro = 30 * 0.009643 / de_libro**2.5 * 16**2.5 / f16 + 70
print(f"Procedimiento de la fuente: d_E = {de_libro:.2f} pg, L_E = {le_libro:.2f} millas de 16 pg   [libro: 19.95, 86.64]")

La fuente usa $\sqrt{f_{14}/f_{16}}$ como peso de **ambas** ramas, cuando cada rama debe llevar $\sqrt{f_E/f_i}$ con
$f_E$ evaluado en el propio $d_E$, y vuelve a mezclar exponentes en la serie. Resultado correcto: 80.3 millas
equivalentes de 16 pg en lugar de 86.6.

### Ejemplo 4.6: línea paralela parcial (*loop*) para aumentar la capacidad

Con la línea del Ejemplo 4.2, ¿qué fracción hay que duplicar con otra de 28 pg para entregar 500 MMpcd adicionales con
las mismas presiones extremas? (Ecs. 4.50-4.51).

In [ ]:
q1 = 1212.557807e6
x = fraccion_loop(q1, q1 + 500e6, 28, 28, 1.0, 1.0)
print(f"x = {x:.4f}  ->  longitud del loop = {x * 41:.2f} millas   [libro: x = 0.6649]")
xs = np.linspace(0, 1, 50)
r = 1.0
q2 = q1 / np.sqrt(1 + xs * (1 / (1 + r)**2 - 1))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs * 41, q2 / 1e6)
ax.axhline((q1 + 500e6) / 1e6, ls="--", c="k", lw=1)
ax.set(xlabel="Longitud duplicada (millas)", ylabel="Gasto (MMpcd)", title="Capacidad vs. longitud del loop")
ax.grid(alpha=0.3); plt.show()